# Restaurant Bills & Tipping: Exploratory Data Analysis (EDA)

**Portfolio project by Swastika Chakraborty**  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy, scikit-learn

## Business question
What patterns are visible in restaurant bills and tips? Do bill size, party size, day, or smoking section relate to tip amount?

## What this notebook demonstrates
- Data inspection and cleaning
- Exploratory data analysis (EDA)
- Outlier detection with the IQR method
- Correlation analysis
- A hypothesis test
- A simple predictive model and evaluation
- Visualizations and plain-English business takeaways

**Dataset note:** This notebook uses Seaborn's public `tips` sample dataset. It is suitable for learning and demonstrating an analysis workflow, but it is small and should not be treated as representative of all restaurants. Interpret associations carefully; correlation does not prove causation.


## 1. Import libraries and load the dataset

Run this notebook from top to bottom in Google Colab or Jupyter Notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Seaborn downloads this small public sample dataset on first use.
df = sns.load_dataset("tips")

print("Dataset loaded successfully.")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
display(df.head())

## 2. Understand the columns and data quality

In [ ]:
print("Column names and data types:")
display(df.dtypes.to_frame("data_type"))

print("\nDataset information:")
df.info()

print("\nMissing values by column:")
display(df.isna().sum().to_frame("missing_values"))

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nNumeric summary:")
display(df.describe().T)

## 3. Clean the data

We preserve the original data in `raw_df`, then remove exact duplicate rows and rows missing key analysis fields. We also check for impossible negative bill or tip values. Document every cleaning decision rather than deleting unusual observations automatically.

In [ ]:
raw_df = df.copy()
df = df.copy()

rows_before = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["total_bill", "tip", "size", "day", "time"])

# Remove impossible negative monetary values or non-positive party sizes.
df = df[(df["total_bill"] >= 0) & (df["tip"] >= 0) & (df["size"] > 0)]

print(f"Rows before cleaning: {rows_before}")
print(f"Rows after cleaning:  {len(df)}")
print(f"Rows removed:         {rows_before - len(df)}")
print("Missing values remaining:")
display(df.isna().sum().to_frame("missing_values"))

## 4. Exploratory data analysis (EDA)

In [ ]:
print("Average total bill:", round(df["total_bill"].mean(), 2))
print("Median total bill:", round(df["total_bill"].median(), 2))
print("Average tip:", round(df["tip"].mean(), 2))
print("Median tip:", round(df["tip"].median(), 2))
print("Average party size:", round(df["size"].mean(), 2))

display(df.groupby("day", observed=False).agg(
    transactions=("total_bill", "size"),
    average_bill=("total_bill", "mean"),
    average_tip=("tip", "mean")
).round(2).sort_values("average_bill", ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="total_bill", kde=True, ax=axes[0])
axes[0].set_title("Distribution of Total Bills")
sns.histplot(data=df, x="tip", kde=True, ax=axes[1])
axes[1].set_title("Distribution of Tips")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="day", y="total_bill")
plt.title("Total Bill Distribution by Day")
plt.xlabel("Day")
plt.ylabel("Total bill")
plt.tight_layout()
plt.show()

## 5. Outlier detection using the IQR method

An outlier is an observation far from the central range. Outliers may be valid large bills, so we flag and investigate them rather than automatically deleting them.

In [ ]:
def iqr_outlier_summary(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flagged = series[(series < lower) | (series > upper)]
    return {
        "Q1": q1, "Q3": q3, "IQR": iqr,
        "lower_bound": lower, "upper_bound": upper,
        "outlier_count": int(flagged.shape[0]),
        "outlier_percent": round(100 * flagged.shape[0] / series.notna().sum(), 2)
    }

outlier_results = pd.DataFrame({
    "total_bill": iqr_outlier_summary(df["total_bill"]),
    "tip": iqr_outlier_summary(df["tip"])
}).T

display(outlier_results.round(2))

bill_info = iqr_outlier_summary(df["total_bill"])
bill_outliers = df[
    (df["total_bill"] < bill_info["lower_bound"]) |
    (df["total_bill"] > bill_info["upper_bound"])
]
print("Flagged high/low bill rows for inspection:")
display(bill_outliers.sort_values("total_bill", ascending=False).head(10))

## 6. Relationships and correlation

In [ ]:
numeric_cols = ["total_bill", "tip", "size"]
corr = df[numeric_cols].corr(numeric_only=True)
display(corr.round(3))

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation Between Bill, Tip, and Party Size")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.regplot(data=df, x="total_bill", y="tip", scatter_kws={"alpha": 0.65})
plt.title("Relationship Between Total Bill and Tip")
plt.tight_layout()
plt.show()

## 7. Hypothesis test

**Question:** In this sample, is the average total bill different between lunch and dinner?

- Null hypothesis (H₀): The population mean bill is the same for lunch and dinner.
- Alternative hypothesis (H₁): The population mean bill differs between lunch and dinner.
- Significance level: 0.05.
- Test: Welch's independent two-sample t-test, which does not assume equal variances.

This is a comparison in the available sample, not proof that meal time causes higher bills.

In [ ]:
lunch = df.loc[df["time"] == "Lunch", "total_bill"].dropna()
dinner = df.loc[df["time"] == "Dinner", "total_bill"].dropna()

t_stat, p_value = stats.ttest_ind(lunch, dinner, equal_var=False)

print(f"Lunch sample size: {len(lunch)}")
print(f"Dinner sample size: {len(dinner)}")
print(f"Mean lunch bill: ${lunch.mean():.2f}")
print(f"Mean dinner bill: ${dinner.mean():.2f}")
print(f"Welch t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.5f}")

alpha = 0.05
if p_value < alpha:
    print("Decision: Reject H0 at the 5% significance level.")
    print("Interpretation: The sample provides statistical evidence of a difference in mean bills.")
else:
    print("Decision: Do not reject H0 at the 5% significance level.")
    print("Interpretation: The sample does not provide sufficient evidence of a difference in mean bills.")

print("Important: statistical significance does not establish causation or business importance.")

## 8. Build and evaluate a simple predictive model

We use a simple linear regression to estimate tip amount from total bill. This is an educational baseline, not a production pricing or staffing model. We use a held-out test set and report MAE, RMSE, and R².

In [ ]:
X = df[["total_bill"]]
y = df["tip"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

plt.figure(figsize=(7, 5))
plt.scatter(y_test, predictions, alpha=0.7)
low = min(y_test.min(), predictions.min())
high = max(y_test.max(), predictions.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual tip")
plt.ylabel("Predicted tip")
plt.title("Actual vs Predicted Tips (Test Set)")
plt.tight_layout()
plt.show()

print("Model coefficient:", round(model.coef_[0], 3))
print("Model intercept:", round(model.intercept_, 3))
print("Interpretation: the fitted slope estimates the average change in predicted tip per one-unit increase in bill, within this sample.")

## 9. Findings for a non-technical audience

Use the computed results above to write a short summary. Avoid unsupported causal statements. Replace the prompts below with your own interpretation after reviewing the outputs.

In [ ]:
# These values are calculated from the cleaned sample; interpret them in context.
bill_tip_corr = df["total_bill"].corr(df["tip"])
largest_day = (
    df.groupby("day", observed=False)["total_bill"]
      .mean()
      .sort_values(ascending=False)
      .index[0]
)
largest_day_mean = df.groupby("day", observed=False)["total_bill"].mean().max()

print("EXECUTIVE SUMMARY")
print("-----------------")
print(f"1. Bill and tip correlation in this sample: {bill_tip_corr:.2f}.")
print("   Explain whether the relationship is weak, moderate, or strong, and note that correlation is not causation.")
print(f"2. The day with the highest average bill in this sample is {largest_day}, at about ${largest_day_mean:.2f}.")
print(f"3. The IQR method flagged {int(bill_info['outlier_count'])} total-bill observations.")
print("   Inspect unusual transactions before deciding whether they are errors or legitimate large bills.")
print(f"4. Lunch-vs-dinner Welch test p-value: {p_value:.5f}.")
print("   State the test decision, sample limitation, and that statistical significance is not the same as business importance.")
print(f"5. On the held-out test set, the simple tip model had MAE {mae:.2f} and R² {r2:.2f}.")
print("   Explain prediction error in plain language and avoid overstating the model's usefulness.")

print("\nYOUR BUSINESS RECOMMENDATIONS (write 2–3 after reviewing the outputs):")
print("- What should a restaurant manager investigate further?")
print("- What additional data would help (e.g., location, promotions, season, service rating)?")
print("- What are the limitations of this small sample?")

## 10. Reproducibility and limitations

- Run all cells from top to bottom in a fresh session.
- Keep cleaning choices and assumptions documented.
- This is a small sample dataset and may not represent other restaurants or time periods.
- The dataset is observational; associations do not prove causation.
- Outlier flags are prompts for investigation, not automatic reasons to delete data.
- The simple regression uses only total bill to estimate tip and is intended as a baseline.
- Before publishing, verify the outputs and write your own conclusions based on the actual results.

## Suggested GitHub repository structure

```text
restaurant-tips-eda/
├── README.md
├── restaurant_tips_eda.ipynb
└── requirements.txt
```

Suggested `requirements.txt`:
```text
pandas
numpy
matplotlib
seaborn
scipy
scikit-learn
jupyter
```

## Suggested resume bullet (use only after you run and verify the notebook)

**Restaurant Bills & Tipping — Exploratory Data Analysis | Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy**
- Cleaned and explored a public restaurant transactions dataset; assessed distributions, flagged IQR-based outliers, visualized correlations, and tested lunch-versus-dinner bill differences using Welch's t-test.
- Evaluated a baseline linear regression model on a held-out test set and documented findings, limitations, and practical questions in a reproducible notebook.
